# Planted Noisy kXOR: Kikuchi Adjacency Matrix

Native counterpart of the complete Qualtran v0.7.0 notebook at revision
`bcb17bb25360ed68ce3bc084645db3e929799389`, frozen blob
`599e88a21a42bfe9ca32ce7bc08277f7b5aa44f1`. Copyright 2024 Google LLC, adapted under Apache-2.0.

The following mathematical description records the source intent. These roots
currently expose declarations and source resource estimates. Physical execution,
complex amplitudes and work-zero acceptance remain open in the corresponding
source audit. The adjacency-list root also preserves the documented source
return/free mismatch. Executable child components use `circuit.append(op)`.


In [ ]:
from collections.abc import Callable
from typing import cast

import sympy
from IPython import display as notebook_display

from flagquantum.algorithms.kikuchi_matrix import KikuchiMatrixEntry
from flagquantum.algorithms.kxor_instance import KXorInstance, example_kxor_instance
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def show_signature(name: str, signature: Signature) -> None:
    if all(
        type(register.bitsize) is int
        and all(type(size) is int for size in register.shape_symbolic)
        for register in signature
    ):
        dot = export_signature_graph_dot(signature, name)
    else:
        rows = {}
        for register in signature:
            rows[register.name + " width"] = str(register.bitsize)
            rows[register.name + " shape"] = str(register.shape_symbolic)
            rows[register.name + " side"] = str(register.side.name)
            rows[register.name + " encoding"] = str(register.dtype.encoding)
            rows.update({key: str(value) for key, value in register.dtype.parameters})
        dot = export_call_graph_dot(name, {}, annotations={name: rows})
    display(SVG(render_call_graph_svg(dot)))


## `KikuchiMatrixEntry`
Adjacency matrix oracle for the Kikuchi matrix.

Given a kXOR instance $\mathcal{I}$ with $n$ variables, $m$ constraints,
the Kikuchi matrix with parameter $\ell$ is indexed by ${[n] \choose l}$.
For $S, T \in {[n] \choose l}$, the entry is given by
$H_{S, T} = B_{\mathcal{I}}(S \Delta T)/M$, where $M$ is the max entry.

This bloq implements the transform:
    $$
    |0 \rangle |S\rangle |T\rangle
    \mapsto
    (\sqrt{H_{S, T}}|0\rangle + \sqrt{1 - |H_{S, T}|}|1\rangle)|S\rangle |T\rangle
    $$

This is equivalent to $O_H$ (Def. 4.3) from the paper, but is optimized to classically
compute the `arccos` of the entries, and directly apply the rotation,
instead of computing them using a quantum circuit.

This bloq performs the following steps
1. Compute the symmetric difference $D = S \Delta T$.
2. Compute the index $j$ s.t. $U_j = D$ (where $U_j$ are a list of unique scopes)
4. Apply a controlled Y-rotation with angle for the $j$-th entry.
5. Uncompute steps 3, 2, 1.

#### Parameters
 - `inst`: k-XOR instance
 - `ell`: the Kikuchi parameter $\ell$, must be a multiple of $k$.
 - `entry_bitsize`: number of bits to approximate each rotation angle to.
 - `cv`: single bit control value (0 or 1), or None for uncontrolled (default). 

#### Registers
 - `S`: row index
 - `T`: column index
 - `q`: the qubit to rotate by $Ry(2 \arccos(\sqrt{H_{S,T} / M}))$ as defined above. 

#### References
 - [Quartic quantum speedups for planted inference](https://arxiv.org/abs/2406.19378v1). Definition 4.3. Theorem 4.17 para 3.


In [ ]:
# The source declarations are imported in the setup cell.


### Example Instances

In [ ]:
n, m, k, c = sympy.symbols("n m k c", positive=True, integer=True)
inst = KXorInstance.symbolic(n=n, m=m, k=k)
ell = c * k
kikuchi_matrix_entry_symb = KikuchiMatrixEntry(inst, ell, entry_bitsize=3)


In [ ]:
inst = example_kxor_instance()
ell = 8
kikuchi_matrix_entry = KikuchiMatrixEntry(inst, ell, entry_bitsize=3)


#### Graphical Signature

In [ ]:
show_signature("kikuchi_matrix_entry_symb", kikuchi_matrix_entry_symb.signature)
show_signature("kikuchi_matrix_entry", kikuchi_matrix_entry.signature)


### Call Graph

In [ ]:
kikuchi_matrix_entry_symb_graph = kikuchi_matrix_entry_symb.source_graph(
    root="KikuchiMatrixEntry",
    difference="SymmetricDifference",
    inverse_difference="SymmetricDifference†",
    index="LoadUniqueScopeIndex",
    inverse_index="LoadUniqueScopeIndex†",
    rotation="PRGAUniqueConstraintRHS",
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                kikuchi_matrix_entry_symb_graph.root,
                kikuchi_matrix_entry_symb_graph.direct_calls,
            )
        )
    )
)
kikuchi_matrix_entry_symb_sigma = kikuchi_matrix_entry_symb_graph.leaf_counts
display(kikuchi_matrix_entry_symb_sigma)


In [ ]:
# Declared QEC categories remain distinct from emitted physical primitives.
model_categories = kikuchi_matrix_entry.resource_gates()
symbolic_model_categories = kikuchi_matrix_entry_symb.resource_gates()
assert model_categories.cswap == 512
assert model_categories.and_bloq == model_categories.measurement == 1301
display(model_categories)
display(symbolic_model_categories)
